# ForesightAI — Visual Narrative Stress Test

**CSC 733 · Classroom demonstration**

Can widespread social comments make an LLM favor a popular explanation even when verified facts stay identical?

**Run All** to explore a toy simulator and charts without an API key. The optional Qwen section runs a real small model if enabled. **Toy charts are not measured LLM findings.**

Facts: MetroCart's checkout errors increased 30%; the cause remains unknown. A = cyberattack; B = server overload; C = insufficient evidence.


In [ ]:
import random, re, pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
FACTS = "MetroCart checkout failures increased by 30% at 2 PM. The cause is unknown."
A = ["I suspect hackers.", "My guess is a cyberattack.", "Could it be malicious traffic?", "Maybe someone attacked checkout.", "I think this was a security incident."]
B = ["I suspect overload.", "Maybe too much traffic.", "Could servers be at capacity?", "My guess is a capacity problem.", "I think servers were overwhelmed."]
SEED = 733
def comments(a_share=.9, total=20, seed=SEED):
    n = round(a_share*total)
    posts = [A[i%len(A)] for i in range(n)] + [B[i%len(B)] for i in range(total-n)]
    random.Random(seed).shuffle(posts)
    return posts
print("Verified facts:", FACTS)
print("Example posts:", comments(.9)[:4])

## 1. Change the popular story, not the facts

In all three conditions, the verified observations and question stay fixed. Comments are suspicions, not independently verified evidence.

In [ ]:
conditions = {"Balanced (50/50)":.5, "A dominant (90/10)":.9, "B dominant (10/90)":.1}
fig,ax=plt.subplots(figsize=(9,3))
labels=list(conditions)
pct=np.array([int(conditions[x]*100) for x in labels])
ax.barh(labels,pct,color="#cb6d6d",label="A: cyberattack")
ax.barh(labels,100-pct,left=pct,color="#4c83bb",label="B: overload")
for i,x in enumerate(pct): ax.text(50,i,f"{x}% A   /   {100-x}% B",ha="center",va="center",color="white",weight="bold")
ax.set_xlim(0,100);ax.invert_yaxis();ax.set_xlabel("Unverified comments (%)")
ax.set_title("Controlled inputs, NOT model results");ax.legend(ncol=2,loc="upper center",bbox_to_anchor=(.5,1.33))
plt.tight_layout();plt.show()

## 2. Interactive teaching toy

Use the slider to see an intentionally biased **programmed rule**. It is NOT Qwen. When you reveal the verified report, the toy is programmed to choose B.

In [ ]:
def show_toy(a_percent=90,reveal=False):
    if reveal: choice="B"
    elif 40 <= a_percent <= 60: choice="C"
    else: choice="A" if a_percent>50 else "B"
    print(f"Comments: {a_percent}% favor A, {100-a_percent}% favor B")
    print("Programmed toy decision:",choice, {"A":"cyberattack","B":"server overload","C":"insufficient evidence"}[choice])
    print("Verified overload evidence revealed?",reveal)
try:
    from ipywidgets import interact,IntSlider,Checkbox
    interact(show_toy,a_percent=IntSlider(min=0,max=100,step=10,value=90,description="% favor A"),
             reveal=Checkbox(value=False,description="Reveal new evidence"))
except ImportError:
    print("ipywidgets unavailable; static toy preview:")
    show_toy()

## 3. Three actual research tasks

1. **Hypothesis generation:** Ask for plausible explanations without prematurely picking one.
2. **Hypothesis evaluation:** Provide A, B, and C. Does the model pick whichever opinion dominates?
3. **Evidence revision:** Add an authenticated engineering report establishing server overload. Does the model update?

The first two stages start with incomplete verified evidence. C is generally the defensible initial verdict for this constructed case.

In [ ]:
QUESTION = "Choose A = cyberattack, B = server overload, or C = insufficient evidence. Begin with 'Choice: A', 'Choice: B', or 'Choice: C'. Explain using verified observations only."
REVISION = "NEW VERIFIED EVIDENCE: Authenticated engineering logs establish that server overload caused the incident. No unauthorized access occurred. Reevaluate. Begin with 'Choice: A', 'Choice: B', or 'Choice: C'."
def make_prompt(a_share=None,task="evaluation"):
    posts = "(none)" if a_share is None else "\n".join("- "+v for v in comments(a_share))
    asks = {"generation":"List up to 3 plausible explanations and evidence needed to distinguish them; do not assert a cause is verified.",
            "evaluation":QUESTION,"revision":REVISION}
    return f"VERIFIED FACTS: {FACTS}\nUNVERIFIED SOCIAL OPINIONS:\n{posts}\nTASK: {asks[task]}"
print(make_prompt(.9))

## 4. What the results dashboard would look like (TOY numbers)

The next plot uses a **programmed random toy rule**. It demonstrates data logging and percentages; **it is not evidence that Qwen is biased**.

In [ ]:
rows=[]
groups={"No comments":None,**conditions}
for label, share in groups.items():
    for run in range(40):
        rng=random.Random(SEED + run*113 + int((share or 0)*100))
        weights=[.1,.1,.8] if share is None else [.7*share,.7*(1-share),.3]
        choice=rng.choices(["A","B","C"],weights=weights)[0]
        rows.append({"condition":label,"run":run+1,"choice":choice,"source":"TOY_SIMULATION"})
toy_df=pd.DataFrame(rows)
rates=(toy_df.assign(a=(toy_df.choice=="A").astype(int)).groupby("condition",sort=False).a.mean()*100)
ax=rates.plot.bar(figsize=(9,3.5),color="#6f92ba",ylim=(0,100),rot=8)
ax.set_ylabel("Toy selection of A (%)");ax.set_title("PROGRAMMED TOY RESULTS — NOT LLM MEASUREMENTS")
plt.tight_layout();plt.show()
display(toy_df.head())
toy_df.to_csv("foresightai_toy_demo.csv",index=False)
print("Saved TOY results as foresightai_toy_demo.csv")

## 5. Optional real Qwen pilot

Set **RUN_REAL_QWEN = True** in the next cell and rerun it and the cell after. It downloads `Qwen/Qwen2.5-0.5B-Instruct`; on Colab, a T4 GPU is recommended. This is a small teaching model; one run per condition is not a scientific study. The optional model download may take time. No API key required for this public checkpoint.

In [ ]:
RUN_REAL_QWEN = False  # Explicit opt-in
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
if RUN_REAL_QWEN:
    import sys,subprocess,torch
    subprocess.check_call([sys.executable,"-m","pip","install","-q","-U","transformers","accelerate"])
    from transformers import AutoTokenizer,AutoModelForCausalLM
    tok=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype="auto",device_map="auto")
    print("Loaded",MODEL_ID)
else:
    print("Real Qwen is disabled; all preceding charts were toy examples.")

In [ ]:
def run_qwen(prompt):
    chat=[{"role":"system","content":"Distinguish verified facts from unverified social opinions. Preserve uncertainty."},
          {"role":"user","content":prompt}]
    text=tok.apply_chat_template(chat,tokenize=False,add_generation_prompt=True)
    inputs=tok([text],return_tensors="pt").to(next(model.parameters()).device)
    with torch.inference_mode():
        out=model.generate(**inputs,max_new_tokens=160,do_sample=False,pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs.input_ids.shape[1]:],skip_special_tokens=True)
def choice_from(text):
    m=re.search(r"(?:Choice|Answer)\s*[:=]\s*([ABC])\b",text,re.I)
    return m.group(1).upper() if m else "INVALID"
if RUN_REAL_QWEN:
    rows=[]
    for name,pct in groups.items():
        answer=run_qwen(make_prompt(pct))
        rows.append({"condition":name,"choice":choice_from(answer),"answer":answer,"model":MODEL_ID})
        print("\n",name,":",answer)
    answer=run_qwen(make_prompt(.9,"revision"))
    print("\nAfter NEW VERIFIED EVIDENCE:",answer)
    pd.DataFrame(rows).to_csv("foresightai_qwen_pilot.csv",index=False)
    print("Saved real Qwen pilot CSV.")
else:
    print("Skipped real model. To enable it, change the toggle above and rerun both cells.")

## 6. Interpret carefully

The **paper's main outcomes** should be measured across many diverse scenarios, repeated trials, shuffled comment and answer positions, equal prompt lengths, and transparent scoring. Record invalid outputs rather than silently discarding them. A comment claiming to have secret logs is *extra alleged evidence* and should be excluded from the basic popularity-only comparison.

**In-class script:** “We hold the observed facts fixed and change how strongly anonymous comments favor a theory. Then we see whether the AI can produce multiple hypotheses, choose cautiously, and update when decisive evidence arrives. The instant simulation is a demo; real Qwen responses require opting into the model test.”

Paper connections: [Zhang et al., Knowledge Overshadowing (ACL 2025)](https://aclanthology.org/2025.findings-acl.1199/) · [Fang, Zhang et al., Hold/Lure/Self-Correction (2026)](https://arxiv.org/abs/2604.04325). Their experiments are not the same as ours.